# NB-02 — めぐ指数: 統合回帰モデル推定

**目的**: ペース・馬場・斤量・レースレベルの4補正係数（β₁〜β₄）と基準タイム（固定効果）を単一の OLS 回帰で一括推定する

**前提**: NB-01 の `megu_dataset.parquet` が生成済みであること

**出力**:
- `megu_regression_params` テーブル（β₁〜β₄ の推定値）
- `megu_par_time` テーブル（距離×コース×芝ダート×馬場カテゴリ別の基準タイム）

**回帰式**:
```
raw_time = β₀
         + β₁ × front_split_dev      # ペース補正係数
         + β₂ × TSI_offset           # 馬場補正係数
         + β₃ × weight_dev × dist_scale  # 斤量補正係数
         + β₄ × log(FQ / par_FQ)    # レースレベル補正係数
         + Σγᵢ × fixed_effect_i      # 距離×コース×芝ダート×馬場カテゴリ
         + ε
```

In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/work/keiba-vpn')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import statsmodels.api as sm
from pathlib import Path

INPUT_DIR  = Path('output/nb01')
OUTPUT_DIR = Path('output/nb02')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(INPUT_DIR / 'megu_dataset.parquet')
print(f'読み込み完了: {len(df):,} 行')
print(df.dtypes)

## 1. 説明変数の作成

In [ ]:
# --- 基準前半スプリット（距離×馬場種別×馬場カテゴリの平均） ---
split_par = (
    df.dropna(subset=['split_time_sec'])
    .groupby(['distance', 'surface', 'track_condition'])['split_time_sec']
    .mean()
    .rename('par_split_time_sec')
)
df = df.join(split_par, on=['distance', 'surface', 'track_condition'])

# front_split_dev: 実前半スプリット - 基準スプリット（正 = スロー）
df['front_split_dev'] = df['split_time_sec'] - df['par_split_time_sec']

# --- 馬場補正: TSI_offset（既存 TrackSpeedIndex から取得） ---
# TSI_offset はデータに既に含まれている前提（なければ別途マージ）
# 正値 = 速い馬場（時計が出やすい日）
if 'tsi_offset' not in df.columns:
    df['tsi_offset'] = 0.0  # フォールバック: 補正なし
    print('WARNING: tsi_offset が見つかりません。Δtrack=0 として処理します。')

# --- 斤量補正変数 ---
STD_WEIGHT_MALE   = 55.0   # 牡・セン基準
STD_WEIGHT_FEMALE = 53.0   # 牝馬基準

df['std_weight'] = np.where(df['sex'] == '牝', STD_WEIGHT_FEMALE, STD_WEIGHT_MALE)
df['weight_dev'] = df['weight_entry'] - df['std_weight']    # 実斤量 - 基準斤量（kg）
df['dist_scale'] = df['distance'] / 2000.0                  # 2000m 基準
df['weight_x_dist'] = df['weight_dev'] * df['dist_scale']   # 斤量補正の合成変数

# --- レースレベル補正変数 ---
par_log_fq = df['log_fq'].mean()  # 全体平均（幾何平均の対数）
df['log_fq_dev'] = df['log_fq'] - par_log_fq   # log(FQ) - log(par_FQ) = log(FQ/par_FQ)

print('=== 説明変数の基本統計 ===')
print(df[['front_split_dev', 'tsi_offset', 'weight_x_dist', 'log_fq_dev']].describe())

In [ ]:
# --- 固定効果ダミー変数 ---
# 距離帯（スプリント/マイル/中距離/長距離）
def distance_band(d):
    if d < 1500:   return 'sprint'
    if d < 1800:   return 'mile'
    if d < 2400:   return 'middle'
    return 'long'

df['distance_band'] = df['distance'].apply(distance_band)

# 固定効果セル = 距離×コース×芝ダート×馬場カテゴリ
# regression formula では C() でカテゴリ変数として扱う
df['fe_cell'] = (
    df['distance'].astype(str) + '_' +
    df['course'] + '_' +
    df['surface'] + '_' +
    df['track_condition']
)

print(f'固定効果セル数: {df["fe_cell"].nunique()}')
print(f'セルあたりの最小サンプル数: {df.groupby("fe_cell").size().min()}')
print(f'30件未満のセル数: {(df.groupby("fe_cell").size() < 30).sum()}')

## 2. OLS 回帰の実行

In [ ]:
# 回帰に使うデータ（必須変数がすべて揃っているレコード）
df_reg = df.dropna(subset=['finish_time_sec', 'fe_cell', 'weight_x_dist', 'log_fq_dev']).copy()

# 前半スプリットが欠損の場合は front_split_dev=0 として回帰に含める
df_reg['front_split_dev'] = df_reg['front_split_dev'].fillna(0)
df_reg['split_available'] = df['split_time_sec'].notna().astype(float)

# FQ が欠損の場合は log_fq_dev=0
df_reg['log_fq_dev'] = df_reg['log_fq_dev'].fillna(0)

print(f'回帰使用データ: {len(df_reg):,} 行')

# OLS 回帰（statsmodels）
formula = (
    'finish_time_sec ~ '
    'front_split_dev + tsi_offset + weight_x_dist + log_fq_dev '
    '+ C(fe_cell)'
)

model = smf.ols(formula, data=df_reg).fit()
print(model.summary())

In [ ]:
# 主要係数の抽出
params = model.params
pvalues = model.pvalues
conf = model.conf_int()

key_params = ['front_split_dev', 'tsi_offset', 'weight_x_dist', 'log_fq_dev']
labels = ['β₁（ペース）', 'β₂（馬場）', 'β₃（斤量）', 'β₄（レースレベル）']

print('=== 主要係数（秒単位）===')
for k, label in zip(key_params, labels):
    if k in params:
        print(f'{label}: coef={params[k]:.4f}  p={pvalues[k]:.4f}  95%CI=[{conf.loc[k,0]:.4f}, {conf.loc[k,1]:.4f}]')

print(f'\nR²: {model.rsquared:.4f}')
print(f'Adj.R²: {model.rsquared_adj:.4f}')
print(f'RMSE: {np.sqrt(model.mse_resid):.3f} 秒')

## 3. 残差分析

In [ ]:
df_reg['residual'] = model.resid
df_reg['fitted']   = model.fittedvalues

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 残差分布
axes[0].hist(df_reg['residual'], bins=80, color='steelblue', alpha=0.8)
axes[0].set_title('残差分布')
axes[0].set_xlabel('残差（秒）')

# フィッテッド vs 残差
axes[1].scatter(df_reg['fitted'], df_reg['residual'], alpha=0.1, s=2)
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_xlabel('フィッテッド値')
axes[1].set_ylabel('残差')
axes[1].set_title('フィッテッド vs 残差')

# QQ プロット
sm.qqplot(df_reg['residual'], line='s', ax=axes[2])
axes[2].set_title('QQ プロット')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'residual_analysis.png', dpi=120)
plt.show()

print(f'残差の標準偏差: {df_reg["residual"].std():.3f} 秒')
print(f'1点= 0.1秒 換算での残差 SD: {df_reg["residual"].std()*10:.1f} ポイント')

## 4. めぐ指数の計算と基準タイムの保存

In [ ]:
# 確定した係数
beta_pace   = params.get('front_split_dev', 0)
beta_track  = params.get('tsi_offset', 0)
beta_weight = params.get('weight_x_dist', 0)
beta_level  = params.get('log_fq_dev', 0)

# 各補正量の計算（秒）
df_reg['delta_pace']   = beta_pace   * df_reg['front_split_dev']
df_reg['delta_track']  = -beta_track * df_reg['tsi_offset']    # 符号反転: 速い馬場=マイナス補正
df_reg['delta_weight'] = beta_weight * df_reg['weight_x_dist']
df_reg['delta_level']  = beta_level  * df_reg['log_fq_dev']

# 補正済みタイム・めぐ指数
df_reg['adjusted_time'] = (
    df_reg['finish_time_sec']
    - df_reg['delta_pace']
    - df_reg['delta_track']
    - df_reg['delta_weight']
    - df_reg['delta_level']
)

# 基準タイム = 固定効果の予測値（補正変数をゼロにした場合のフィッテッド値）
df_reg['par_time'] = (
    df_reg['fitted']
    - beta_pace   * df_reg['front_split_dev']
    - beta_track  * df_reg['tsi_offset']
    - beta_weight * df_reg['weight_x_dist']
    - beta_level  * df_reg['log_fq_dev']
)

df_reg['megu_index'] = 100 + (df_reg['par_time'] - df_reg['adjusted_time']) * 10

print('=== めぐ指数の分布 ===')
print(df_reg['megu_index'].describe())
df_reg['megu_index'].hist(bins=80, color='coral', alpha=0.8)
plt.xlabel('めぐ指数')
plt.title('めぐ指数の全体分布')
plt.axvline(100, color='red', linestyle='--', label='par=100')
plt.legend()
plt.savefig(OUTPUT_DIR / 'megu_index_distribution.png', dpi=120)
plt.show()

In [ ]:
# 基準タイムを距離×コース×芝ダート×馬場カテゴリ別に集計
par_time_table = (
    df_reg.groupby(['distance', 'course', 'surface', 'track_condition'])
    .agg(
        par_time_sec=('par_time', 'mean'),
        par_front_split_sec=('par_split_time_sec', 'mean'),
        sample_count=('finish_time_sec', 'count')
    )
    .reset_index()
)

print(f'基準タイムセル数: {len(par_time_table):,}')
print(par_time_table.head(10))
par_time_table.to_parquet(OUTPUT_DIR / 'megu_par_time.parquet', index=False)

In [ ]:
# DB への保存
from src.db.session import get_session, init_engine
from sqlalchemy.dialects.postgresql import insert as pg_insert
from src.db.models import MeguRegressionParams, MeguParTime
import datetime

MODEL_VERSION = 'v1'
init_engine()

param_rows = [
    {'param_name': 'beta_pace',   'param_value': float(beta_pace),   'model_version': MODEL_VERSION},
    {'param_name': 'beta_track',  'param_value': float(beta_track),  'model_version': MODEL_VERSION},
    {'param_name': 'beta_weight', 'param_value': float(beta_weight), 'model_version': MODEL_VERSION},
    {'param_name': 'beta_level',  'param_value': float(beta_level),  'model_version': MODEL_VERSION},
    {'param_name': 'par_log_fq',  'param_value': float(par_log_fq),  'model_version': MODEL_VERSION},
]

with get_session() as session:
    stmt = pg_insert(MeguRegressionParams).values(param_rows)
    stmt = stmt.on_conflict_do_update(
        constraint='megu_regression_params_param_name_model_version_key',
        set_={'param_value': stmt.excluded.param_value}
    )
    session.execute(stmt)
    session.commit()

print(f'回帰係数 {len(param_rows)} 件を DB に保存しました')

In [ ]:
# めぐ指数結果を保存（NB-05/06 で使用）
df_reg[['race_id', 'horse_id', 'finish_time_sec', 'par_time',
        'delta_pace', 'delta_track', 'delta_weight', 'delta_level',
        'adjusted_time', 'megu_index']].to_parquet(
    OUTPUT_DIR / 'megu_index_results.parquet', index=False
)
print('めぐ指数計算結果を保存しました')

# 係数サマリー
print('\n=== 確定係数サマリー ===')
print(f'β₁（ペース補正）  : {beta_pace:.4f} 秒/秒  → 前半1秒遅いと全体で {beta_pace:.3f}秒補正')
print(f'β₂（馬場補正）    : {beta_track:.4f} 秒  → TSI 1秒速い日で {beta_track:.3f}秒補正')
print(f'β₃（斤量補正）    : {beta_weight:.4f} 秒/kg  → 2000mで1kg重いと {beta_weight:.3f}秒補正 (理論値: 0.2)')
print(f'β₄（レースレベル）: {beta_level:.4f} 秒/log → FQ が2倍になると {beta_level*np.log(2):.3f}秒補正')